In [ ]:
import requests
import pandas as pd
import time
import os

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36",
    "Referer": "https://www.musinsa.com/",
    "Origin": "https://www.musinsa.com",
}
BRAND         = "filluminate"
CATEGORY      = "002"
CATEGORY_NAME = "아우터"
PAGE_SIZE     = 100 # 추측 최대 상한선이 100
#001 : 상의, 002 : 아우터, 003 : 하의

# CSV 파일 경로
PRODUCTS_CSV = f"products_{CATEGORY}.csv"
REVIEWS_CSV  = f"reviews_{CATEGORY}.csv"
SURVEY_CSV   = f"survey_{CATEGORY}.csv"
ERRORS_CSV   = f"errors_{CATEGORY}.csv"

session = requests.Session()
session.headers.update(headers)
session.headers["Referer"] = f"https://www.musinsa.com/brand/{BRAND}"

def request_json(url, params=None, timeout=10, retry=3):
    last_status = None
    last_error = None
    for attempt in range(1, retry + 1):
        try:
            response = session.get(url, params=params, timeout=timeout)
            last_status = response.status_code
            if response.status_code == 200:
                return response.json(), response.status_code
            print(f"요청 실패 ({response.status_code}) -> 재시도 {attempt}/{retry}")
        except Exception as e:
            last_error = e
            print(f"요청 오류 -> 재시도 {attempt}/{retry}: {e}")
        time.sleep(2 * attempt)
    if last_error is not None:
        print(f"최종 요청 오류: {last_error}")
    return None, last_status

# ==========================================
# Step 1: 브랜드 전체 상품 수집
# ==========================================
all_data = []
page = 1
while True:
    try:
        url = "https://api.musinsa.com/api2/dp/v2/plp/goods"
        params = {
            "gf":       "A",
            "sortCode": "REVIEW",
            "category": CATEGORY,
            "brand":    BRAND,
            "size":     PAGE_SIZE,
            "caller":   "FLAGSHIP",
            "page":     page,
        }
        json_data, status_code = request_json(url, params=params, timeout=10, retry=3)
        if not json_data:
            print(f"상품 수집 실패 (page {page}): {status_code}")
            break
        data = json_data.get("data", {})
        goods_list = data.get("list", [])
        if not goods_list:
            break
        for item in goods_list:
            all_data.append({
                "플랫폼":    "무신사",
                "카테고리":  CATEGORY_NAME,
                "브랜드":    item.get("brandName", ""),
                "goodsNo":  str(item.get("goodsNo", "")),
                "상품명":    item.get("goodsName", ""),
                "정가":      item.get("normalPrice", 0),
                "판매가":    item.get("price", 0),
                "할인율(%)": item.get("saleRate", 0),
                "리뷰수":    item.get("reviewCount", 0),
                "리뷰점수":  item.get("reviewScore", 0),
            })
        total_pages = data.get("pagination", {}).get("totalPages", 1)
        print(f"상품 수집 중... {page}/{total_pages} 페이지 (누적 {len(all_data)}개)")
        if page >= total_pages:
            break
        page += 1
        time.sleep(2)
    except Exception as e:
        print(f"상품 수집 오류 (page {page}): {e}")
        time.sleep(2)
        break
# 리뷰 많은 순으로 정렬 (전체)
df = pd.DataFrame(all_data)
df = df.sort_values("리뷰수", ascending=False).reset_index(drop=True)
df["조회수"]    = 0
df["누적판매수"] = 0
print(f"\n상품 총 {len(df)}개 수집 완료 (리뷰 많은 순)")
print(df[["카테고리", "브랜드", "상품명", "리뷰수"]].to_string(index=False))

# ==========================================
# Step 2: 재시작 로직 - 이미 수집된 goodsNo 및 리뷰번호 확인
# ==========================================
collected_goods   = set()
collected_review_ids = set() # 상품 단위가 아닌 '리뷰 번호'를 추적하기 위한 set

if os.path.exists(PRODUCTS_CSV):
    df_existing = pd.read_csv(PRODUCTS_CSV)
    collected_goods = set(df_existing["goodsNo"].astype(str).tolist())
    print(f"\n기존 수집된 상품 {len(collected_goods)}개 확인 -> 정보 업데이트 스킵 예정")
else:
    print("\n기존 수집 상품 파일 없음 -> 처음부터 수집 시작")

if os.path.exists(REVIEWS_CSV):
    df_existing_reviews = pd.read_csv(REVIEWS_CSV)
    collected_review_ids = set(df_existing_reviews["리뷰번호"].astype(str).tolist())
    print(f"기존 수집된 개별 리뷰 {len(collected_review_ids)}개 확인 -> 중복 리뷰 스킵 예정")

# ==========================================
# Step 3: 만족도 파싱 함수
# ==========================================
def parse_satisfaction(survey):
    if not survey:
        return ""
    questions = survey.get("questions", [])
    parts = []
    for q in questions:
        attribute = q.get("attribute", "")
        answers = q.get("answers", [])
        answer_text = ", ".join([a.get("answerShortText", "") for a in answers])
        parts.append(f"{attribute}: {answer_text}")
    return " / ".join(parts)

# ==========================================
# Step 4: 리뷰 수집 함수 (새로운 리뷰만 수집)
# ==========================================
def get_reviews(goods_no, target_count, collected_ids, max_pages=9999):
    reviews = []
    stop_crawling = False # 크롤링 중단 플래그

    for page in range(max_pages):
        if len(reviews) >= target_count or stop_crawling:
            break
        
        url = "https://goods.musinsa.com/api2/review/v1/view/list"
        params = {
            "page":              page,
            "pageSize":          10,
            "goodsNo":           goods_no,
            "sort":              "new", # 최신순 정렬 (중요)
            "selectedSimilarNo": goods_no,
            "myFilter":          "false",
            "hasPhoto":          "false",
            "isExperience":      "false",
        }
        try:
            json_data, status_code = request_json(url, params=params, timeout=10, retry=3)
            if not json_data:
                print(f"  상태코드 {status_code} -> 중단")
                break
                
            data        = json_data.get("data", {})
            review_list = data.get("list", [])
            
            if not review_list:
                break
                
            for review in review_list:
                review_no = str(review.get("no", ""))
                
                # 핵심 로직: 이미 수집된 리뷰 번호를 만나면 더 이상 과거 리뷰를 탐색할 필요가 없음
                if review_no in collected_ids:
                    stop_crawling = True
                    break 
                
                profile = review.get("userProfileInfo") or {}
                images  = review.get("images") or []
                reviews.append({
                    "goodsNo":  str(goods_no),
                    "리뷰번호": review_no, # 이 값이 중복 체크의 기준이 됩니다
                    "작성자":   profile.get("userNickName", ""),
                    "리뷰내용": review.get("content", ""),
                    "평점":     int(review.get("grade", 0)),
                    "체험단":   review.get("type") == "experience",
                    "구매옵션": review.get("goodsOption", ""),
                    "키":       profile.get("userHeight", ""),
                    "몸무게":   profile.get("userWeight", ""),
                    "성별":     profile.get("reviewSex", ""),
                    "작성일":   review.get("createDate", ""),
                    "만족도":   parse_satisfaction(review.get("reviewSurveySatisfaction")),
                    "사진유무": len(images) > 0,
                    "도움돼요": review.get("likeCount", 0),
                })
                
                if len(reviews) >= target_count:
                    break
                    
            total_pages = data.get("page", {}).get("totalPages", 0)
            if page >= total_pages - 1:
                break
                
            if len(reviews) % 100 == 0 and len(reviews) > 0:
                print(f"  새로운 리뷰 {len(reviews)}개 수집 중...")
            time.sleep(2)
            
        except Exception as e:
            print(f"  리뷰 수집 오류: {e}")
            time.sleep(2)
            break
            
    return reviews

# ==========================================
# Step 5: 상품 통계(조회수, 판매량) 수집 함수
# ==========================================
def get_goods_stats(goods_no):
    url = f"https://goods-detail.musinsa.com/api2/goods/{goods_no}/stat"
    try:
        json_data, status_code = request_json(url, timeout=10, retry=3)
        if json_data:
            sales_count = json_data.get("data", {}).get("purchaseTotal", 0)
            views_count = json_data.get("data", {}).get("pageViewTotal", 0)
            return sales_count, views_count
        else:
            print(f"  통계 수집 실패 ({goods_no}): {status_code}")
    except Exception as e:
        print(f"  통계 수집 오류 ({goods_no}): {e}")
    return 0, 0

# ==========================================
# Step 6: survey 수집 함수
# ==========================================
def get_survey(goods_no):
    url = f"https://goods.musinsa.com/api2/review/v1/view/survey/{goods_no}/summary"
    try:
        json_data, status_code = request_json(url, timeout=10, retry=3)
        if json_data:
            questions = json_data.get("data", {}).get("questions", [])
            rows = []
            for q in questions:
                attribute = q.get("attribute", "")
                for answer in q.get("answers", []):
                    rows.append({
                        "goodsNo": str(goods_no),
                        "항목":    attribute,
                        "답변":    answer.get("answerShortText", ""),
                        "비율(%)": answer.get("percentage", 0),
                        "응답수":  answer.get("count", 0),
                    })
            return rows
        else:
            print(f"  survey 수집 실패 ({goods_no}): {status_code}")
    except Exception as e:
        print(f"  survey 수집 오류 ({goods_no}): {e}")
    return []

# ==========================================
# Step 7: 중간 저장 함수
# ==========================================
def save_product(row_dict):
    df_row = pd.DataFrame([row_dict])
    if os.path.exists(PRODUCTS_CSV):
        df_row.to_csv(PRODUCTS_CSV, mode="a", header=False, index=False, encoding="utf-8-sig")
    else:
        df_row.to_csv(PRODUCTS_CSV, mode="w", header=True, index=False, encoding="utf-8-sig")
def save_reviews(reviews):
    if not reviews:
        return
    df_r = pd.DataFrame(reviews)
    if os.path.exists(REVIEWS_CSV):
        df_r.to_csv(REVIEWS_CSV, mode="a", header=False, index=False, encoding="utf-8-sig")
    else:
        df_r.to_csv(REVIEWS_CSV, mode="w", header=True, index=False, encoding="utf-8-sig")
def save_survey(survey_rows):
    if not survey_rows:
        return
    df_s = pd.DataFrame(survey_rows)
    if os.path.exists(SURVEY_CSV):
        df_s.to_csv(SURVEY_CSV, mode="a", header=False, index=False, encoding="utf-8-sig")
    else:
        df_s.to_csv(SURVEY_CSV, mode="w", header=True, index=False, encoding="utf-8-sig")
def save_error(goods_no, error_type, message):
    df_e = pd.DataFrame([{
        "goodsNo":    goods_no,
        "error_type": error_type,
        "message":    message,
        "time":       pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
    }])
    if os.path.exists(ERRORS_CSV):
        df_e.to_csv(ERRORS_CSV, mode="a", header=False, index=False, encoding="utf-8-sig")
    else:
        df_e.to_csv(ERRORS_CSV, mode="w", header=True, index=False, encoding="utf-8-sig")
        
# ==========================================
# Step 8: 전체 상품 리뷰 및 통계 수집
# ==========================================
total_items = len(df)
for idx, row in df.iterrows():
    goods_no     = row["goodsNo"]
    target_count = int(row["리뷰수"])
    current_item = idx + 1
    product_saved = goods_no in collected_goods

    print(f"\n[{current_item}/{total_items}] 처리 중: {str(row['상품명'])[:25]} (총 리뷰 수: {target_count}개)")

    sales, views = 0, 0

    if not product_saved:
        # 통계 수집
        try:
            sales, views = get_goods_stats(goods_no)
        except Exception as e:
            print(f"  통계 수집 실패: {e}")
            save_error(goods_no, "stats", str(e))
            sales, views = 0, 0
        time.sleep(2)

        # survey 수집
        try:
            survey_rows = get_survey(goods_no)
            save_survey(survey_rows)
            print(f"  survey {len(survey_rows)}행 저장 완료")
        except Exception as e:
            print(f"  survey 수집 실패: {e}")
            save_error(goods_no, "survey", str(e))
        time.sleep(2)

        # 상품 정보 저장
        save_product({
            "플랫폼":    row["플랫폼"],
            "카테고리":  row["카테고리"],
            "브랜드":    row["브랜드"],
            "goodsNo":  goods_no,
            "상품명":    row["상품명"],
            "정가":      row["정가"],
            "판매가":    row["판매가"],
            "할인율(%)": row["할인율(%)"],
            "조회수":    views,
            "누적판매수": sales,
            "리뷰수":    row["리뷰수"],
            "리뷰점수":  row["리뷰점수"],
        })
        collected_goods.add(goods_no)
        print(f"  -> 상품 저장 완료 (누적판매: {sales} | 조회수: {views})")
    else:
        print("  [스킵] 상품 기본 정보는 이미 저장됨")

    time.sleep(2)

    # 리뷰 수집 (무조건 체크)
    if not goods_no or target_count == 0:
        print(f"  [스킵] 리뷰 없음")
    else:
        print(f"  새로운 리뷰 수집 확인 중...")
        try:
            # collected_review_ids를 넘겨주어 중복을 체크하게 합니다.
            reviews = get_reviews(goods_no, target_count=target_count, collected_ids=collected_review_ids)
            
            if len(reviews) > 0:
                save_reviews(reviews)
                # 새로 수집된 리뷰 번호들을 기존 set에 업데이트
                collected_review_ids.update([r["리뷰번호"] for r in reviews])
                print(f"  -> 새로운 리뷰 {len(reviews)}개 수집 및 추가 완료!")
            else:
                print(f"  -> 추가된 새로운 리뷰가 없습니다.")
                
        except Exception as e:
            print(f"  리뷰 수집 실패: {e}")
            save_error(goods_no, "review", str(e))
            
    time.sleep(2)

In [ ]:
import pandas as pd

# CSV 파일 불러오기
df_reviews = pd.read_csv("reviews_002.csv")
df_products = pd.read_csv("products_002.csv")

# goodsNo별로 몇 개의 리뷰(행)가 있는지 카운트 (내림차순 자동 정렬)
review_counts = df_reviews["goodsNo"].value_counts()

# 결과 출력
print(review_counts)

# 작성일 최근 일자 확인
print(df_reviews['작성일'].max())